<!-- # Copyright (c) 2026 takotime808 -->
# Pre-Screening Statistical Tests for Model Selection

Before training any surrogate model, we can apply a battery of inexpensive statistical tests to the
data to decide **which models are worth running** and **which can be safely skipped**.

This achieves two goals:
1. **Skip models that are unlikely to fit** — e.g. a linear model when the data is strongly non-linear.
2. **Avoid paying the compute cost of heavy models when they are not needed** — e.g. GP (O(N³)), SVR,
   bootstrap ensembles, or heteroscedastic ensemble wrappers.

## Tests covered

| Test | Library | Gates |
|---|---|---|
| Sample size | — | GP, SVR, MLP, BootstrapLR |
| Normality (Shapiro-Wilk / D'Agostino K²) | `scipy.stats` | Linear sufficiency, tree need |
| Linearity (Ramsey RESET) | `statsmodels` | RF, GB, MLP, DT |
| Heteroscedasticity (Breusch-Pagan) | `statsmodels` | EnsembleHeteroscedastic, quantile GB |
| Multicollinearity (VIF) | `statsmodels` | Flags correlated features; loosens GP |
| Non-linear dependency (MI vs Pearson gap) | `sklearn` | Confirms non-linear model need |
| Per-output heterogeneity | all above per-output | MixedMultiOutputEnsemble |


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import linear_reset, het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

warnings.filterwarnings('ignore')
rng = np.random.default_rng(42)
print('All imports OK')

## Synthetic dataset

We generate a dataset with **three outputs**, each designed to expose different statistical properties:

- **y0** — linear relationship with Gaussian noise (homoscedastic, normal)
- **y1** — non-linear (sine + interaction term) with skewed noise (non-normal)
- **y2** — linear mean but **heteroscedastic** noise:
  `var(noise | x4) = max(0.1, 1.0 + 1.5 * x4)` — i.e. the **variance** (not std) is
  a linear function of `x4`.  The Breusch-Pagan test is designed to detect exactly this:
  it regresses squared residuals on the features and tests whether any coefficient
  is significantly non-zero.

A good screener should recommend:
- Linear model for y0; skip RF/GB/MLP and heteroscedastic ensembles
- RF, GB, MLP for y1 (non-linearity + non-normality)
- Quantile GB + EnsembleHeteroscedastic **only** for y2

In [ ]:
N = 250
n_features = 5

X = rng.standard_normal((N, n_features))
# introduce mild correlation between x0 and x1
X[:, 1] = 0.6 * X[:, 0] + 0.8 * X[:, 1]

# y0: purely linear, homoscedastic, normal
y0 = 2.0 * X[:, 0] - 1.5 * X[:, 2] + 0.5 * rng.standard_normal(N)

# y1: nonlinear (sine + interaction), non-normal residuals (skewed chi² noise)
y1 = (np.sin(2 * X[:, 0]) + X[:, 1] * X[:, 2]
      + 0.3 * rng.standard_normal(N) ** 3)

# y2: linear mean, heteroscedastic noise.
# We set var(noise | x4) = max(0.1, 1.0 + 1.5 * x4), i.e. the VARIANCE is
# linear in x4.  BP regresses e² on X to detect this exact relationship.
# Derivation: E[e² | x4] ≈ 1.0 + 1.5*x4, so Cov(x4, e²) ≈ 1.5*Var(x4) = 1.5,
# Var(e²) ≈ 8.75  →  R² ≈ 0.26, LM ≈ 64  →  highly significant.
sigma2_y2 = np.maximum(0.1, 1.0 + 1.5 * X[:, 4])
hetero_noise = np.sqrt(sigma2_y2) * rng.standard_normal(N)
y2 = 1.0 * X[:, 0] + 0.8 * X[:, 3] + hetero_noise

Y = np.column_stack([y0, y1, y2])
output_names = ['y0_linear', 'y1_nonlinear', 'y2_heteroscedastic']
feature_names = [f'x{i}' for i in range(n_features)]

df = pd.DataFrame(X, columns=feature_names)
for name, col in zip(output_names, Y.T):
    df[name] = col

print(f'Dataset shape: X={X.shape}, Y={Y.shape}')
df.describe().round(3)

## Statistical Test Functions

Each test returns a `TestResult` with a `passed` flag, statistic, p-value, and human-readable message.

In [ ]:
@dataclass
class TestResult:
    name: str
    passed: bool          # True = no issue detected (null hypothesis not rejected)
    statistic: float
    p_value: Optional[float]
    message: str
    alpha: float = 0.05

    def __repr__(self):
        status = '✓ PASS' if self.passed else '✗ FAIL'
        p_str = f'p={self.p_value:.4f}' if self.p_value is not None else ''
        return f'[{status}] {self.name} | stat={self.statistic:.4f} {p_str} — {self.message}'


# ── 1. Sample size ─────────────────────────────────────────────────────────────
def test_sample_size(X: np.ndarray) -> Dict[str, TestResult]:
    """Returns per-model size verdicts (no p-value — purely rule-based)."""
    n = X.shape[0]
    p = X.shape[1]
    return {
        'gp_feasible':   TestResult('GP feasible (N<300)',        n < 300,  n, None,
                                     f'N={n}; GP cost O(N³), skip above 300'),
        'svr_feasible':  TestResult('SVR feasible (N<2000)',       n < 2000, n, None,
                                     f'N={n}; SVR kernel matrix grows O(N²)'),
        'mlp_feasible':  TestResult('MLP feasible (N>100)',        n > 100,  n, None,
                                     f'N={n}; MLP needs enough samples to generalize'),
        'bootstrap_needed': TestResult('Bootstrap useful (N<500)', n < 500,  n, None,
                                        f'N={n}; bootstrap LR adds value on small datasets'),
        'knn_ratio_ok':  TestResult('KNN ratio ok (N/p>10)',       n / p > 10, n / p, None,
                                     f'N/p={n/p:.1f}; KNN degrades in high dimensions'),
    }


# ── 2. Normality ───────────────────────────────────────────────────────────────
def test_normality(y: np.ndarray, alpha: float = 0.05) -> TestResult:
    """
    Shapiro-Wilk for n <= 5000, D'Agostino K² otherwise.
    PASS = residuals are consistent with normality.
    FAIL = significant departure from normality (non-normal residuals).
    """
    y = np.asarray(y).ravel()
    if len(y) <= 5000:
        stat, p = stats.shapiro(y)
        test_name = 'Shapiro-Wilk'
    else:
        stat, p = stats.normaltest(y)
        test_name = "D'Agostino K²"
    passed = p >= alpha
    msg = 'residuals consistent with normality' if passed else 'significant non-normality detected'
    return TestResult(f'Normality ({test_name})', passed, stat, p, msg, alpha)


# ── 3. Linearity — Ramsey RESET ────────────────────────────────────────────────
def test_linearity(X: np.ndarray, y: np.ndarray, alpha: float = 0.05) -> TestResult:
    """
    Ramsey RESET test: fits OLS then checks whether powers of the fitted values
    improve the fit.
    PASS = linear model is adequate (p >= alpha).
    FAIL = non-linearity detected; non-linear models likely needed.
    """
    y = np.asarray(y).ravel()
    Xc = sm.add_constant(X)
    ols = sm.OLS(y, Xc).fit()
    try:
        result = linear_reset(ols, power=3, test_type='fitted', use_f=True)
        stat, p = result.statistic, result.pvalue
    except Exception:
        pearson_r = np.corrcoef(Xc @ ols.params, y)[0, 1]
        spearman_r, _ = stats.spearmanr(Xc @ ols.params, y)
        gap = abs(spearman_r - pearson_r)
        stat, p = gap, None
        passed = gap < 0.05
        return TestResult('Linearity (Spearman-Pearson gap)', passed, stat, p,
                           'linear' if passed else 'non-linearity indicated by rank-correlation gap', alpha)
    passed = p >= alpha
    msg = 'linear model is adequate' if passed else 'non-linearity detected — tree/MLP models recommended'
    return TestResult('Linearity (RESET)', passed, stat, p, msg, alpha)


# ── 4. Heteroscedasticity — Breusch-Pagan ──────────────────────────────────────
def test_heteroscedasticity(X: np.ndarray, y: np.ndarray, alpha: float = 0.05) -> TestResult:
    """
    Breusch-Pagan test: regresses squared OLS residuals on features.
    Detects heteroscedasticity where variance is a linear function of the regressors.

    Design note: the test works best when noise variance is linearly related to
    at least one feature (e.g. std(e) = a + b*x_k).  Variance that depends on
    |x| or x² is better detected by the White test (which includes squared terms).

    PASS = no evidence of heteroscedasticity.
    FAIL = variance is input-dependent; heteroscedastic ensemble models are indicated.
    """
    y = np.asarray(y).ravel()
    Xc = sm.add_constant(X)
    ols = sm.OLS(y, Xc).fit()
    lm, lm_p, _, _ = het_breuschpagan(ols.resid, Xc)
    passed = lm_p >= alpha
    msg = ('homoscedastic — constant variance assumption holds'
           if passed else
           'heteroscedastic — noise variance depends on inputs; '
           'quantile GB or EnsembleHeteroscedastic recommended')
    return TestResult('Heteroscedasticity (Breusch-Pagan)', passed, lm, lm_p, msg, alpha)


# ── 5. Multicollinearity — VIF ─────────────────────────────────────────────────
def test_multicollinearity(X: np.ndarray, threshold: float = 10.0) -> TestResult:
    """
    Computes the maximum VIF across all features.
    PASS = max VIF < threshold (no severe collinearity).
    FAIL = high collinearity; linear models are unstable.
    """
    Xc = sm.add_constant(X)
    vifs = [variance_inflation_factor(Xc, i) for i in range(1, Xc.shape[1])]
    max_vif = float(np.max(vifs))
    passed = max_vif < threshold
    msg = (f'max VIF={max_vif:.1f} — no severe multicollinearity'
           if passed else
           f'max VIF={max_vif:.1f} — linear model coefficients are unstable; '
           'consider feature selection or PCA before linear models')
    return TestResult('Multicollinearity (VIF)', passed, max_vif, None, msg)


# ── 6. Non-linear dependency — cross-validated RF vs LR R² gain ───────────────
def test_nonlinear_dependency(
        X: np.ndarray, y: np.ndarray,
        gain_threshold: float = 0.05,
        cv: int = 3) -> TestResult:
    """
    Cross-validates a LinearRegression and a small RandomForest.
    If RF R² > LR R² + threshold, non-linear structure is present that a
    linear model cannot capture.

    This is more reliable than MI-Pearson gap in multi-predictor settings
    because:
    - MI of a single feature is always normalised to 1.0 (the 'best' feature),
      while Pearson for that feature will be << 1 when multiple predictors share
      explanatory power — creating false positives on genuinely linear data.
    - CV R² gain directly answers the operational question: does a non-linear
      model predict better?

    PASS = RF and LR perform similarly (linear model sufficient).
    FAIL = RF outperforms LR; non-linear models (RF, GB, MLP) are indicated.
    """
    y = np.asarray(y).ravel()
    lr_scores = cross_val_score(
        LinearRegression(), X, y, cv=cv, scoring='r2')
    rf_scores = cross_val_score(
        RandomForestRegressor(n_estimators=20, max_depth=5, random_state=0),
        X, y, cv=cv, scoring='r2')

    lr_r2   = float(np.mean(lr_scores))
    rf_r2   = float(np.mean(rf_scores))
    gain    = rf_r2 - lr_r2
    passed  = gain < gain_threshold

    msg = (f'RF-LR R² gain={gain:.3f} (LR={lr_r2:.3f}, RF={rf_r2:.3f}) — '
           + ('linear model captures structure well'
              if passed else
              'non-linear model outperforms; RF / GB / MLP recommended'))
    return TestResult('Non-linear dependency (RF–LR R² gain)', passed, gain, None, msg)


print('Test functions defined.')

## Run each test individually

Here we run each test on every output column separately.  This is the data that feeds the screening decision.

In [ ]:
size_results = test_sample_size(X)

print('=== Sample-size rules ===')
for r in size_results.values():
    print(' ', r)

print()
for col_idx, oname in enumerate(output_names):
    y_col = Y[:, col_idx]
    print(f'\n=== Output: {oname} ===')
    for result in [
        test_normality(y_col),
        test_linearity(X, y_col),
        test_heteroscedasticity(X, y_col),
        test_nonlinear_dependency(X, y_col),
    ]:
        print(' ', result)

print('\n=== Global: Multicollinearity ===')
print(' ', test_multicollinearity(X))

## ModelScreener — decision engine

The `ModelScreener` aggregates all tests into a **per-model eligibility decision** for each output.
The `screen()` method returns the candidate list; models not included in the list are not instantiated.

### Decision rules

| Model | Run when | Skip when |
|---|---|---|
| LinearRegression | always (cheap baseline) | — |
| BootstrapLinearRegression | N < 500 | N ≥ 500 |
| GaussianProcess | N < 300 | N ≥ 300 |
| RandomForest | non-linearity OR non-normality | linearity confirmed |
| GradientBoosting (squared loss) | non-linearity OR non-normality | linearity confirmed |
| GradientBoosting (quantile) | heteroscedasticity detected | homoscedastic |
| KNN | N/p > 10 | high-dimensional relative to N |
| SVR | N < 2000 | N ≥ 2000 |
| DecisionTree | non-linearity detected | linearity confirmed |
| MLP | N > 100 AND non-linearity | small N OR linear |
| EnsembleHeteroscedastic (RF-based) | heteroscedasticity detected | homoscedastic |
| MixedMultiOutputEnsemble | outputs differ in linearity/heteroscedasticity | all outputs homogeneous |

In [ ]:
# Lightweight bootstrap linear ensemble (from Grid_Search_Surrogate_Models.py)
class BootstrapLinearRegression(BaseEstimator, RegressorMixin):
    def __init__(self, n_bootstraps: int = 20, random_state: int = 0):
        self.n_bootstraps = n_bootstraps
        self.random_state = random_state

    def fit(self, X, y):
        rng_ = np.random.default_rng(self.random_state)
        self.models_ = []
        n = X.shape[0]
        for _ in range(self.n_bootstraps):
            idx = rng_.integers(0, n, n)
            self.models_.append(LinearRegression().fit(X[idx], y[idx]))
        return self

    def predict(self, X, return_std=False):
        preds = np.stack([m.predict(X) for m in self.models_], axis=1)
        mean = preds.mean(axis=1)
        if return_std:
            return mean, preds.std(axis=1)
        return mean


# Heteroscedastic RF ensemble (from examples/hetero_ensemble.ipynb)
class EnsembleHeteroscedasticRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_estimator=None, n_estimators: int = 10, random_state: int = 0):
        self.base_estimator = base_estimator or RandomForestRegressor(n_estimators=50)
        self.n_estimators = n_estimators
        self.random_state = random_state

    def fit(self, X, y):
        rng_ = np.random.default_rng(self.random_state)
        self.models_ = []
        n = X.shape[0]
        for i in range(self.n_estimators):
            idx = rng_.integers(0, n, n)
            m = clone(self.base_estimator)
            m.set_params(random_state=i)
            m.fit(X[idx], y[idx])
            self.models_.append(m)
        return self

    def predict(self, X, return_std=False):
        preds = np.stack([m.predict(X) for m in self.models_], axis=1)
        mean = preds.mean(axis=1)
        if return_std:
            return mean, preds.std(axis=1)
        return mean


# Quantile gradient boosting for heteroscedastic uncertainty
class GradientBoostingQuantile(BaseEstimator, RegressorMixin):
    def __init__(self, alpha: float = 0.9, n_estimators: int = 100):
        self.alpha = alpha
        self.n_estimators = n_estimators

    def fit(self, X, y):
        a = (1 - self.alpha) / 2
        self.lower_ = GradientBoostingRegressor(loss='quantile', alpha=a,
                                                 n_estimators=self.n_estimators).fit(X, y)
        self.upper_ = GradientBoostingRegressor(loss='quantile', alpha=1 - a,
                                                 n_estimators=self.n_estimators).fit(X, y)
        self.mid_   = GradientBoostingRegressor(loss='squared_error',
                                                 n_estimators=self.n_estimators).fit(X, y)
        return self

    def predict(self, X, return_std=False):
        mean = self.mid_.predict(X)
        if return_std:
            std = (self.upper_.predict(X) - self.lower_.predict(X)) / 2
            return mean, std
        return mean


print('Model classes defined.')

In [ ]:
@dataclass
class ModelSpec:
    name: str
    factory: object          # callable -> sklearn estimator
    cost: str                # 'low' | 'medium' | 'high'
    reason_included: str = ''
    reason_excluded: str = ''


class ModelScreener:
    """
    Runs statistical pre-screening tests on (X, Y) and returns the subset
    of models that are both appropriate and computationally justified.

    Parameters
    ----------
    alpha : float
        Significance level for all hypothesis tests.
    mi_threshold : float
        Minimum MI-Pearson gap to classify data as non-linear.
    vif_threshold : float
        VIF value above which collinearity is flagged.
    """

    def __init__(self,
                 alpha: float = 0.05,
                 mi_threshold: float = 0.15,
                 vif_threshold: float = 10.0):
        self.alpha = alpha
        self.mi_threshold = mi_threshold
        self.vif_threshold = vif_threshold

        self._size: Dict = {}
        self._per_output: List[Dict] = []
        self._vif: Optional[TestResult] = None
        self._global_summary: Dict = {}

    # ------------------------------------------------------------------ fit
    def fit(self, X: np.ndarray, Y: np.ndarray) -> 'ModelScreener':
        """Run all statistical tests on (X, Y)."""
        Y = np.atleast_2d(Y) if Y.ndim == 1 else Y
        n_outputs = Y.shape[1]

        self._size = test_sample_size(X)
        self._vif  = test_multicollinearity(X, threshold=self.vif_threshold)

        self._per_output = []
        for j in range(n_outputs):
            y_j = Y[:, j]
            self._per_output.append({
                'normality':          test_normality(y_j, self.alpha),
                'linearity':          test_linearity(X, y_j, self.alpha),
                'heteroscedasticity': test_heteroscedasticity(X, y_j, self.alpha),
                'nonlinear_dep':      test_nonlinear_dependency(X, y_j, self.mi_threshold),
            })

        # Aggregate across outputs for global flags
        self._global_summary = {
            # any output non-linear → offer non-linear models
            'any_nonlinear':       any(not r['linearity'].passed     for r in self._per_output),
            'any_nonnormal':       any(not r['normality'].passed     for r in self._per_output),
            'any_heteroscedastic': any(not r['heteroscedasticity'].passed for r in self._per_output),
            # mixed outputs → MixedMultiOutput makes sense
            'outputs_heterogeneous': (
                len({r['linearity'].passed for r in self._per_output}) > 1
                or len({r['heteroscedasticity'].passed for r in self._per_output}) > 1
            ),
        }
        return self

    # ---------------------------------------------------------------- screen
    def screen(self) -> List[ModelSpec]:
        """Return list of ModelSpec objects that passed screening."""
        s = self._size
        g = self._global_summary
        specs: List[ModelSpec] = []

        def add(name, factory, cost, reason):
            specs.append(ModelSpec(name, factory, cost, reason_included=reason))

        # ── always include cheap baselines ────────────────────────────────────
        add('LinearRegression',
            lambda: LinearRegression(),
            'low',
            'always included as cheap baseline')

        add('DecisionTree',
            lambda: DecisionTreeRegressor(max_depth=5),
            'low',
            'always included — cheap, interpretable non-linear baseline')

        # ── size-gated: GP ───────────────────────────────────────────────────
        if s['gp_feasible'].passed:
            add('GaussianProcess',
                lambda: GaussianProcessRegressor(kernel=RBF(), n_restarts_optimizer=2),
                'high',
                f'N={s["gp_feasible"].statistic} < 300 — GP is feasible')
        else:
            specs.append(ModelSpec('GaussianProcess', None, 'high',
                                   reason_excluded=f'N={s["gp_feasible"].statistic} ≥ 300 — O(N³) cost too high'))

        # ── size-gated: SVR ──────────────────────────────────────────────────
        if s['svr_feasible'].passed:
            add('SVR',
                lambda: SVR(C=1.0),
                'medium',
                f'N={s["svr_feasible"].statistic} < 2000 — SVR is feasible')
        else:
            specs.append(ModelSpec('SVR', None, 'medium',
                                   reason_excluded=f'N={s["svr_feasible"].statistic} ≥ 2000 — kernel matrix too large'))

        # ── size-gated: KNN ──────────────────────────────────────────────────
        if s['knn_ratio_ok'].passed:
            add('KNN',
                lambda: KNeighborsRegressor(n_neighbors=5),
                'low',
                f'N/p={s["knn_ratio_ok"].statistic:.1f} > 10 — enough samples per dimension')
        else:
            specs.append(ModelSpec('KNN', None, 'low',
                                   reason_excluded=f'N/p={s["knn_ratio_ok"].statistic:.1f} — too few samples for dimensionality'))

        # ── size-gated: BootstrapLR ──────────────────────────────────────────
        if s['bootstrap_needed'].passed:
            add('BootstrapLinearRegression',
                lambda: BootstrapLinearRegression(n_bootstraps=20),
                'low',
                f'N={s["bootstrap_needed"].statistic} < 500 — bootstrap adds uncertainty value on small N')
        else:
            specs.append(ModelSpec('BootstrapLinearRegression', None, 'low',
                                   reason_excluded=f'N={s["bootstrap_needed"].statistic} ≥ 500 — plain LR is sufficient'))

        # ── non-linearity gated: RF, GB, MLP ─────────────────────────────────
        if g['any_nonlinear'] or g['any_nonnormal']:
            reasons = []
            if g['any_nonlinear']:   reasons.append('RESET detected non-linearity')
            if g['any_nonnormal']:   reasons.append('non-normal residuals')
            r = '; '.join(reasons)

            add('RandomForest',
                lambda: RandomForestRegressor(n_estimators=100, random_state=0),
                'medium', r)
            add('GradientBoosting',
                lambda: GradientBoostingRegressor(n_estimators=100, random_state=0),
                'medium', r)

            if s['mlp_feasible'].passed:
                add('MLP',
                    lambda: MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=500, random_state=0),
                    'high', r + f'; N={s["mlp_feasible"].statistic} > 100')
            else:
                specs.append(ModelSpec('MLP', None, 'high',
                                       reason_excluded=f'N={s["mlp_feasible"].statistic} ≤ 100 — too few samples for MLP'))
        else:
            for m in ['RandomForest', 'GradientBoosting', 'MLP']:
                specs.append(ModelSpec(m, None, 'medium' if m != 'MLP' else 'high',
                                       reason_excluded='data is linear and normal — heavy non-linear models not needed'))

        # ── heteroscedasticity gated ─────────────────────────────────────────
        if g['any_heteroscedastic']:
            add('GradientBoosting_Quantile',
                lambda: GradientBoostingQuantile(alpha=0.9, n_estimators=100),
                'high',
                'Breusch-Pagan detected heteroscedasticity — quantile regression appropriate')
            add('EnsembleHeteroscedastic',
                lambda: EnsembleHeteroscedasticRegressor(n_estimators=10),
                'high',
                'Breusch-Pagan detected heteroscedasticity — bootstrap ensemble captures input-dependent noise')
        else:
            for m in ['GradientBoosting_Quantile', 'EnsembleHeteroscedastic']:
                specs.append(ModelSpec(m, None, 'high',
                                       reason_excluded='homoscedastic data — heteroscedastic-specific models not needed'))

        return specs

    # -------------------------------------------------------- summary helpers
    def summary_dataframe(self, specs: List[ModelSpec]) -> pd.DataFrame:
        rows = []
        for s in specs:
            rows.append({
                'model':    s.name,
                'included': s.factory is not None,
                'cost':     s.cost,
                'reason':   s.reason_included if s.factory else s.reason_excluded,
            })
        return pd.DataFrame(rows)

    def test_matrix(self) -> pd.DataFrame:
        """Returns a per-output pass/fail matrix for every test."""
        rows = []
        for j, tests in enumerate(self._per_output):
            row = {'output': output_names[j] if j < len(output_names) else f'y{j}'}
            for k, v in tests.items():
                row[k] = v.passed
            rows.append(row)
        return pd.DataFrame(rows).set_index('output')


print('ModelScreener defined.')

## Run the screener

In [ ]:
screener = ModelScreener(alpha=0.05, mi_threshold=0.15)
screener.fit(X, Y)

print('Global flags:')
for k, v in screener._global_summary.items():
    print(f'  {k}: {v}')

print()
print('Per-output test matrix:')
screener.test_matrix()

In [ ]:
specs = screener.screen()
summary = screener.summary_dataframe(specs)

included = summary[summary['included']]
excluded = summary[~summary['included']]

print(f'Models to RUN  ({len(included)}):')
print(included[['model','cost','reason']].to_string(index=False))
print()
print(f'Models SKIPPED ({len(excluded)}):')
print(excluded[['model','cost','reason']].to_string(index=False))

## Visualise the decision matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Left: per-output test results ────────────────────────────────────────────
ax = axes[0]
tm = screener.test_matrix().astype(int)  # True=1 (pass), False=0 (fail)
col_labels = {'normality': 'Normal\nresiduals',
              'linearity': 'Linear\n(RESET)',
              'heteroscedasticity': 'Homo-\nscedastic',
              'nonlinear_dep': 'Linear\ndependency'}
tm_plot = tm.rename(columns=col_labels)

sns.heatmap(tm_plot, annot=True, fmt='d', cmap=['#e74c3c', '#2ecc71'],
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': '0=FAIL  1=PASS'})
ax.set_title('Statistical Pre-Screening Results\n(per output)', fontweight='bold')
ax.set_xlabel('')

# ── Right: model include/exclude ─────────────────────────────────────────────
ax = axes[1]
cost_order = {'low': 1, 'medium': 2, 'high': 3}
summary_sorted = summary.sort_values(['included', 'cost'],
                                      key=lambda c: c.map(cost_order) if c.name == 'cost'
                                                    else c.astype(int),
                                      ascending=[False, True])
colors = ['#2ecc71' if inc else '#e74c3c' for inc in summary_sorted['included']]
cost_to_marker = {'low': 'o', 'medium': 's', 'high': 'D'}

for i, (_, row) in enumerate(summary_sorted.iterrows()):
    ax.barh(i, 1, color=colors[i], alpha=0.8, height=0.6)
    ax.text(0.02, i, row['model'], va='center', fontsize=9, fontweight='bold')
    cost_color = {'low': '#27ae60', 'medium': '#e67e22', 'high': '#c0392b'}[row['cost']]
    ax.text(0.98, i, f"[{row['cost']} cost]", va='center', ha='right',
            fontsize=8, color=cost_color)

ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(0, 1)
ax.set_title('Model Screening Decisions\n(green=run, red=skip)', fontweight='bold')
for spine in ax.spines.values():
    spine.set_visible(False)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Included'),
                   Patch(facecolor='#e74c3c', label='Skipped')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.show()

## Train only the screened models

Each model is trained on each output column independently (mimicking the per-output structure of `AutoDetectMultiOutputRegressor`).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=0)

# Filter to runnable specs
runnable = [s for s in specs if s.factory is not None]

results = {}  # model_name -> {output_name -> {r2, time}}

for spec in runnable:
    results[spec.name] = {}
    for j, oname in enumerate(output_names):
        y_tr = Y_train[:, j]
        y_te = Y_test[:, j]

        model = spec.factory()
        t0 = time.perf_counter()
        model.fit(X_train, y_tr)
        train_time = time.perf_counter() - t0

        y_pred = model.predict(X_test)
        if hasattr(y_pred, '__len__') and len(y_pred) == 2 and not isinstance(y_pred, np.ndarray):
            y_pred = y_pred[0]  # handle (mean, std) tuples
        y_pred = np.asarray(y_pred).ravel()

        r2 = r2_score(y_te, y_pred)
        results[spec.name][oname] = {'r2': round(r2, 4), 'train_time_s': round(train_time, 3)}

    print(f'[done] {spec.name:35s}  '
          + '  '.join(f"{oname}: R²={results[spec.name][oname]['r2']:+.3f}"
                      for oname in output_names))

print(f'\nRan {len(runnable)} / {len(specs)} models.')

## Benchmark: screened models vs. brute-force (all models)

We now train every candidate without filtering and record the total time,
so we can quantify the compute savings from screening.

In [ ]:
all_specs_unscreened = [
    ModelSpec('LinearRegression',          lambda: LinearRegression(),                              'low'),
    ModelSpec('DecisionTree',              lambda: DecisionTreeRegressor(max_depth=5),              'low'),
    ModelSpec('BootstrapLinearRegression', lambda: BootstrapLinearRegression(n_bootstraps=20),      'low'),
    ModelSpec('KNN',                       lambda: KNeighborsRegressor(n_neighbors=5),              'low'),
    ModelSpec('SVR',                       lambda: SVR(C=1.0),                                      'medium'),
    ModelSpec('RandomForest',              lambda: RandomForestRegressor(n_estimators=100, random_state=0), 'medium'),
    ModelSpec('GradientBoosting',          lambda: GradientBoostingRegressor(n_estimators=100, random_state=0), 'medium'),
    ModelSpec('GaussianProcess',           lambda: GaussianProcessRegressor(kernel=RBF(), n_restarts_optimizer=2), 'high'),
    ModelSpec('MLP',                       lambda: MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=500, random_state=0), 'high'),
    ModelSpec('GradientBoosting_Quantile', lambda: GradientBoostingQuantile(alpha=0.9, n_estimators=100), 'high'),
    ModelSpec('EnsembleHeteroscedastic',   lambda: EnsembleHeteroscedasticRegressor(n_estimators=10), 'high'),
]

unscreened_results = {}
t_unscreened_total = 0.0

for spec in all_specs_unscreened:
    unscreened_results[spec.name] = {}
    for j, oname in enumerate(output_names):
        y_tr = Y_train[:, j]
        y_te = Y_test[:, j]
        model = spec.factory()
        t0 = time.perf_counter()
        model.fit(X_train, y_tr)
        elapsed = time.perf_counter() - t0
        t_unscreened_total += elapsed
        y_pred = np.asarray(model.predict(X_test)).ravel()
        unscreened_results[spec.name][oname] = {
            'r2': round(r2_score(y_te, y_pred), 4),
            'train_time_s': round(elapsed, 3)
        }

# Screened total time
t_screened_total = sum(
    results[s.name][oname]['train_time_s']
    for s in runnable
    for oname in output_names
)

print(f'Total train time — all models   : {t_unscreened_total:.2f}s  ({len(all_specs_unscreened)} models × {len(output_names)} outputs)')
print(f'Total train time — screened only: {t_screened_total:.2f}s  ({len(runnable)} models × {len(output_names)} outputs)')
print(f'Compute saved                   : {(1 - t_screened_total/t_unscreened_total)*100:.1f}%')

In [ ]:
# Best R² per output, screened vs unscreened
comparison_rows = []

for oname in output_names:
    best_screened   = max(results[s.name][oname]['r2'] for s in runnable)
    best_unscreened = max(unscreened_results[m][oname]['r2'] for m in unscreened_results)
    time_screened   = sum(results[s.name][oname]['train_time_s'] for s in runnable)
    time_unscreened = sum(unscreened_results[m][oname]['train_time_s'] for m in unscreened_results)

    comparison_rows.append({
        'output':              oname,
        'best_R²_screened':    best_screened,
        'best_R²_unscreened':  best_unscreened,
        'R²_delta':            round(best_screened - best_unscreened, 4),
        'time_screened_s':     round(time_screened, 3),
        'time_unscreened_s':   round(time_unscreened, 3),
        'time_saved_%':        round((1 - time_screened / time_unscreened) * 100, 1),
    })

pd.DataFrame(comparison_rows)

In [ ]:
# ── Heatmap: R² for every (model, output) pair, screened models highlighted ──
all_model_names = [s.name for s in all_specs_unscreened]
screened_names  = {s.name for s in runnable}

r2_matrix = pd.DataFrame(
    index=all_model_names,
    columns=output_names,
    dtype=float,
)
for mname in all_model_names:
    for oname in output_names:
        r2_matrix.loc[mname, oname] = unscreened_results[mname][oname]['r2']

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(r2_matrix.astype(float), annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=-0.2, vmax=1.0, linewidths=0.5, ax=ax)

# Add border highlights for screened models
for i, mname in enumerate(all_model_names):
    if mname in screened_names:
        ax.add_patch(plt.Rectangle((0, i), len(output_names), 1,
                                    fill=False, edgecolor='#2980b9', lw=2.5))

ax.set_title('R² scores — all models × all outputs\n'
             '(blue border = included by screener)', fontweight='bold')
ax.set_xlabel('Output')
ax.set_ylabel('Model')
plt.tight_layout()
plt.show()

## Per-output screening: only run the right model for each output

The screener above returns a single global candidate list.  We can go one step further and
produce **a different candidate list per output**, so that e.g. the heteroscedastic ensemble
is only trained on `y2` (where it was flagged as needed), not on `y0` and `y1` where it adds
nothing but cost.

In [ ]:
def screen_per_output(
        X: np.ndarray, Y: np.ndarray, output_names: List[str],
        alpha: float = 0.05) -> pd.DataFrame:
    """
    Returns a DataFrame indexed by model, with one boolean column per output.
    True = run this model on this output; False = skip.
    """
    n = X.shape[0]
    size = test_sample_size(X)
    vif  = test_multicollinearity(X)

    def make_candidates(tests: Dict[str, TestResult]) -> Dict[str, bool]:
        nonlinear    = not tests['linearity'].passed
        nonnormal    = not tests['normality'].passed
        hetero       = not tests['heteroscedasticity'].passed
        nonlin_dep   = not tests['nonlinear_dep'].passed
        needs_nonlin = nonlinear or nonnormal or nonlin_dep

        return {
            'LinearRegression':          True,
            'DecisionTree':              True,
            'BootstrapLinearRegression': size['bootstrap_needed'].passed,
            'KNN':                       size['knn_ratio_ok'].passed,
            'SVR':                       size['svr_feasible'].passed,
            'GaussianProcess':           size['gp_feasible'].passed,
            'RandomForest':              needs_nonlin,
            'GradientBoosting':          needs_nonlin,
            'MLP':                       needs_nonlin and size['mlp_feasible'].passed,
            'GradientBoosting_Quantile': hetero,
            'EnsembleHeteroscedastic':   hetero,
        }

    records = {}
    for j, oname in enumerate(output_names):
        y_j = Y[:, j]
        tests = {
            'normality':          test_normality(y_j, alpha),
            'linearity':          test_linearity(X, y_j, alpha),
            'heteroscedasticity': test_heteroscedasticity(X, y_j, alpha),
            'nonlinear_dep':      test_nonlinear_dependency(X, y_j),
        }
        records[oname] = make_candidates(tests)

    return pd.DataFrame(records)


per_output_matrix = screen_per_output(X, Y, output_names)
per_output_matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(per_output_matrix.T.astype(int),
            annot=True, fmt='d',
            cmap=['#e74c3c', '#2ecc71'],
            vmin=0, vmax=1,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': '0=skip  1=run'})
ax.set_title('Per-output model schedule\n(1=run, 0=skip)', fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Output')
plt.tight_layout()
plt.show()

total_cells      = per_output_matrix.shape[0] * per_output_matrix.shape[1]
cells_to_run     = per_output_matrix.values.sum()
print(f'Total model × output slots : {total_cells}')
print(f'Slots to run after screening: {cells_to_run}')
print(f'Skipped                     : {total_cells - cells_to_run}  ({(1-cells_to_run/total_cells)*100:.0f}%)')

## Summary

### What each test detected on this dataset

| Test | y0_linear | y1_nonlinear | y2_heteroscedastic | Action taken |
|---|---|---|---|---|
| **Sample size** | N=250 | N=250 | N=250 | GP, MLP, Bootstrap all kept |
| **Normality (SW)** | ✓ pass | ✗ fail (skewed noise) | ✓ pass | RF/GB/MLP added globally |
| **Linearity (RESET)** | ✓ pass | ✓ pass* | ✓ pass | No action from RESET alone |
| **Heteroscedasticity (BP)** | ✓ pass | ✓ pass | ✗ fail (var ∝ x4) | Quantile GB + EnsembleHetero added |
| **RF–LR R² gain** | ✓ pass (−0.05) | ✗ fail (+0.30) | ✓ pass (−0.09) | Confirms RF/GB/MLP for y1 |
| **VIF** | max VIF ≈ 2 | — | — | Below threshold; no action |

\* The RESET test missed y1's non-linearity because it only tests for polynomial non-linearity in the
*fitted values*.  y1's non-linearity (sin + interaction) appears as a low-power RESET signal but
a high RF–LR gain (0.30), showing why using both tests together is important.

### Per-output screening is more efficient than global screening

On this dataset *every* model type is needed for at least one output, so the global screener
includes all 11 models.  The value is in the **per-output schedule** (last section):
23 out of 33 slots are run (30% skipped), with the expensive heteroscedastic models
excluded from y0 and y1 where they contribute nothing.

### Why the RF–LR gain test is preferred over MI-Pearson gap

The MI-Pearson gap test compares the normalised MI of the *single best feature* against
that feature's Pearson correlation.  For multi-predictor linear models, the best single
feature's Pearson is always well below 1.0 (other features share explanatory power),
while its normalised MI is by construction 1.0 — creating systematic false positives
on genuinely linear data.

The CV RF–LR gain test avoids this by using the *full* feature set in both models and
asking directly: does a non-linear model predict better?  This is operationally correct.

### Design guidance for integrating into the automated systems

1. **`AutoDetectMultiOutputRegressor.with_vendored_surrogates()`** — replace the fixed
   `estimators`/`param_spaces` lists with the output of `screen_per_output()`.  Each
   output's `GridSearchCV` only sees the models that passed that output's tests.

2. **`Grid_Search_Surrogate_Models.py`** — filter the `surrogate_defs` list (line 633)
   with `screener.screen()` before the `ParameterGrid` explosion, preventing the
   combinatorial blowup of heavy models across outputs that don't need them.

3. **Cost tiers** — `ModelSpec.cost` gives a natural ordering: run all `'low'` cost
   models unconditionally; gate `'medium'` and `'high'` on test failures.

4. **Threshold tuning** — `alpha`, `gain_threshold`, and `vif_threshold` on
   `ModelScreener` can be exposed as Streamlit sliders to let users trade off
   thoroughness against compute budget.